In [1]:
from google import genai
from google.colab import userdata

api_key = userdata.get('Gemini_API_Key')
client = genai.Client(api_key=api_key)

In [2]:
# Função auxiliar para fazer chamadas à API do Gemini
def obter_resposta_llm(texto_prompt, modelo="gemini-3.6-flash"):
    if not client:
        return "Erro: O cliente da API não foi inicializado."

    try:
        resposta = client.models.generate_content(
            model=modelo,
            contents=texto_prompt,
        )
        return resposta.text
    except Exception as e:
        return f"Erro na comunicação com a API do Gemini: {e}"


In [7]:
import re

def calcular(expressao):
    """Ferramenta real: executa um cálculo matemático de verdade."""
    try:
        resultado = eval(expressao, {"__builtins__": {}}, {})
        return str(resultado)
    except Exception as e:
        return f"Erro ao calcular: {e}"


def loop_react_simples(tarefa, max_passos=4, modelo="gemini-3.6-flash"):
    prompt = f"""Resolva a tarefa abaixo. Você tem acesso a uma ferramenta de cálculo.

Tarefa: {tarefa}

Regras importantes:
- Escreva APENAS um passo por vez (um Pensamento + uma Ação) e PARE.
- Não invente o resultado da Ação — eu vou te devolver a Observação real.
- Quando tiver a resposta final, escreva apenas "Resposta Final: ..."

Formato de um passo:
Pensamento: [seu raciocínio]
Ação: calcular("expressão matemática aqui")
"""

    historico = prompt

    for passo in range(max_passos):
        resposta = obter_resposta_llm(historico, modelo=modelo)
        print(f"--- Passo {passo+1} ---\n{resposta}\n")

        if "Resposta Final" in resposta:
            return resposta

        match_acao = re.search(r'calcular\("(.+?)"\)', resposta)
        if match_acao:
            expressao = match_acao.group(1)
            observacao = calcular(expressao)
            historico += f"\n{resposta}\nObservação: {observacao}\n"
            print(f"Observação real: {observacao}\n")
        else:
            return resposta  # modelo não seguiu o formato, encerra

    return "Número máximo de passos atingido."


# Teste com uma tarefa simples
tarefa = "Quanto é 15% de 240, somado com 8% de 150?"
resultado = loop_react_simples(tarefa)
print(f"\n=== RESULTADO FINAL ===\n{resultado}")

--- Passo 1 ---
Pensamento: Vou calcular o valor de 15% de 240 somado com 8% de 150 usando uma expressão matemática direta.
Ação: calcular("0.15 * 240 + 0.08 * 150")

Observação real: 48.0

--- Passo 2 ---
Resposta Final: 48


=== RESULTADO FINAL ===
Resposta Final: 48
